#Install Required Libraries

In [52]:
!pip install -q -U \
    chromadb \
    sentence-transformers \
    gradio \
    wikipedia-api \
    tavily-python \
    google-genai \
    langchain-text-splitters \
    "opentelemetry-api<=1.42.1" \
    "opentelemetry-sdk<=1.42.1"

#Import Required Libraries


In [53]:
import os
import json

import chromadb
import wikipediaapi
import gradio as gr

from sentence_transformers import SentenceTransformer

print("✅ All libraries imported successfully.")

✅ All libraries imported successfully.


#Configure Wikipedia API


In [54]:
wiki = wikipediaapi.Wikipedia(
    user_agent="SportsQuizAgent/1.0 (your-email@example.com)",
    language="en"
)

print("✅ Wikipedia API Ready")

✅ Wikipedia API Ready


#Define Sports Topics


In [55]:
sports_topics = {
    "Cricket": [
        "Cricket",
        "ICC Cricket World Cup",
        "Indian Premier League",
        "Test cricket",
        "Twenty20 cricket"
    ],

    "Football": [
        "Association football",
        "FIFA World Cup",
        "UEFA Champions League",
        "Premier League"
    ],

    "Tennis": [
        "Tennis",
        "Grand Slam (tennis)",
        "Wimbledon Championships",
        "ATP Tour"
    ],

    "Badminton": [
        "Badminton",
        "Thomas Cup",
        "Uber Cup",
        "BWF World Championships"
    ],

    "Basketball": [
        "Basketball",
        "National Basketball Association",
        "FIBA Basketball World Cup",
        "Olympic basketball"
    ]
}

print("✅ Sports Topics Loaded")

✅ Sports Topics Loaded


#Fetch Wikipedia Articles

    

In [56]:
def fetch_wikipedia_articles(topic_map):
    documents = []

    for sport, topics in topic_map.items():
        for topic in topics:
            page = wiki.page(topic)

            if page.exists():
                documents.append({
                    "sport": sport,
                    "title": page.title,
                    "text": page.text,
                    "url": page.fullurl
                })

                print(f"✅ {page.title}")
            else:
                print(f"❌ {topic} not found")

    return documents

#Download Sports Knowledge


In [57]:
sports_documents = fetch_wikipedia_articles(sports_topics)

print(f"\n📚 Total Articles Collected: {len(sports_documents)}")

✅ Cricket
✅ Cricket World Cup
✅ Indian Premier League
✅ Test cricket
✅ Twenty20
✅ Association football
✅ FIFA World Cup
✅ UEFA Champions League
✅ Premier League
✅ Tennis
✅ Grand Slam (tennis)
✅ Wimbledon Championships
✅ ATP Tour
✅ Badminton
✅ Thomas Cup
✅ Uber Cup
✅ BWF World Championships
✅ Basketball
✅ National Basketball Association
✅ FIBA Basketball World Cup
✅ Basketball at the Summer Olympics

📚 Total Articles Collected: 21


#Validate Collected Documents


In [58]:
print("Total documents:", len(sports_documents))
print("\nAvailable sports:")

for sport in sorted(set(doc["sport"] for doc in sports_documents)):
    count = sum(1 for doc in sports_documents if doc["sport"] == sport)
    print(f"✅ {sport}: {count} articles")

Total documents: 21

Available sports:
✅ Badminton: 4 articles
✅ Basketball: 4 articles
✅ Cricket: 5 articles
✅ Football: 4 articles
✅ Tennis: 4 articles


#Preview One Wikipedia Document


In [59]:
sample_document = sports_documents[0]

print("Sport:", sample_document["sport"])
print("Title:", sample_document["title"])
print("URL:", sample_document["url"])
print("\nText Preview:\n")
print(sample_document["text"][:1000])

Sport: Cricket
Title: Cricket
URL: https://en.wikipedia.org/wiki/Cricket

Text Preview:

Cricket is a bat-and-ball game that is played between two teams of eleven players on a field, at the centre of which is a 22-yard (20-metre; 66-foot) pitch with a wicket at each end, each comprising two bails (small sticks) balanced on three stumps. Two players from the batting team, the striker and nonstriker, stand in front of either wicket holding bats, while one player from the fielding team, the bowler, bowls the ball toward the striker's wicket from the opposite end of the pitch. The striker's goal is to hit the bowled ball with the bat and then switch places with the nonstriker, with the batting team scoring one run for each of these swaps. Runs are also scored when the ball reaches the boundary of the field or when the ball is bowled illegally.
The fielding team aims to prevent runs by dismissing batters (so they are "out"). Dismissal can occur in various ways, including being bowled (when 

#Remove Empty and Very Short Documents


In [60]:
clean_documents = []

for document in sports_documents:
    text = document["text"].strip()

    if len(text) >= 500:
        clean_documents.append({
            "sport": document["sport"],
            "title": document["title"],
            "text": text,
            "url": document["url"]
        })
    else:
        print(f"⚠️ Skipped short article: {document['title']}")

print(f"\n✅ Valid documents: {len(clean_documents)}")


✅ Valid documents: 21


#Text Cleaning Function


In [61]:
import re

def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\[\d+\]", "", text)
    text = text.strip()

    return text

for document in clean_documents:
    document["text"] = clean_text(document["text"])

print("✅ Text cleaning completed")

✅ Text cleaning completed


#Chunk Wikipedia Articles

In [62]:
def create_chunks(text, chunk_size=1200, overlap=200):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if end < len(text):
            last_period = chunk.rfind(".")
            if last_period > 600:
                end = start + last_period + 1
                chunk = text[start:end]

        chunk = chunk.strip()

        if len(chunk) >= 200:
            chunks.append(chunk)

        start = max(end - overlap, start + 1)

    return chunks

#Create Chunks with Metadata


In [63]:
sports_chunks = []

for document in clean_documents:
    chunks = create_chunks(document["text"])

    for index, chunk in enumerate(chunks):
        sports_chunks.append({
            "id": f"{document['sport']}_{document['title']}_{index}",
            "sport": document["sport"],
            "title": document["title"],
            "url": document["url"],
            "chunk_index": index,
            "text": chunk
        })

print(f"✅ Total chunks created: {len(sports_chunks)}")

✅ Total chunks created: 814


#Inspect a Sample Chunk


In [64]:
sample_chunk = sports_chunks[0]

print("Sport:", sample_chunk["sport"])
print("Title:", sample_chunk["title"])
print("Chunk Index:", sample_chunk["chunk_index"])
print("Source:", sample_chunk["url"])
print("\nChunk Text:\n")
print(sample_chunk["text"])

Sport: Cricket
Title: Cricket
Chunk Index: 0
Source: https://en.wikipedia.org/wiki/Cricket

Chunk Text:

Cricket is a bat-and-ball game that is played between two teams of eleven players on a field, at the centre of which is a 22-yard (20-metre; 66-foot) pitch with a wicket at each end, each comprising two bails (small sticks) balanced on three stumps. Two players from the batting team, the striker and nonstriker, stand in front of either wicket holding bats, while one player from the fielding team, the bowler, bowls the ball toward the striker's wicket from the opposite end of the pitch. The striker's goal is to hit the bowled ball with the bat and then switch places with the nonstriker, with the batting team scoring one run for each of these swaps. Runs are also scored when the ball reaches the boundary of the field or when the ball is bowled illegally. The fielding team aims to prevent runs by dismissing batters (so they are "out"). Dismissal can occur in various ways, including bei

#Save Cleaned Dataset and Chunks


In [65]:
with open("sports_wikipedia_data.json", "w", encoding="utf-8") as file:
    json.dump(clean_documents, file, ensure_ascii=False, indent=2)

with open("sports_chunks.json", "w", encoding="utf-8") as file:
    json.dump(sports_chunks, file, ensure_ascii=False, indent=2)

print("✅ Clean documents saved")
print("✅ Sports chunks saved")

✅ Clean documents saved
✅ Sports chunks saved


#Load Embedding Model


In [66]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model loaded successfully


#Prepare Data for Batch Embedding


In [67]:
chunk_ids = [chunk["id"] for chunk in sports_chunks]
chunk_texts = [chunk["text"] for chunk in sports_chunks]

chunk_metadata = [
    {
        "sport": chunk["sport"],
        "title": chunk["title"],
        "url": chunk["url"],
        "chunk_index": chunk["chunk_index"]
    }
    for chunk in sports_chunks
]

print("✅ Data prepared for embeddings")
print("Total chunks:", len(chunk_texts))

✅ Data prepared for embeddings
Total chunks: 814


#Generate Embeddings in Batch


In [68]:
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("✅ Embeddings generated successfully")
print("Embedding shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

✅ Embeddings generated successfully
Embedding shape: (814, 384)


#Create Persistent ChromaDB Collection


In [69]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="./sports_chromadb"
)

collection = chroma_client.get_or_create_collection(
    name="sports_knowledge",
    metadata={"hnsw:space": "cosine"}
)

print("✅ ChromaDB collection ready")
print("Existing vectors:", collection.count())

✅ ChromaDB collection ready
Existing vectors: 1001


#Store Embeddings in ChromaDB

In [70]:
batch_size = 100

for start in range(0, len(chunk_ids), batch_size):
    end = start + batch_size

    collection.upsert(
        ids=chunk_ids[start:end],
        documents=chunk_texts[start:end],
        embeddings=chunk_embeddings[start:end].tolist(),
        metadatas=chunk_metadata[start:end]
    )

    print(
        f"✅ Stored chunks "
        f"{start + 1} to {min(end, len(chunk_ids))}"
    )

print("\n✅ All chunks stored in ChromaDB")
print("Total vectors:", collection.count())

✅ Stored chunks 1 to 100
✅ Stored chunks 101 to 200
✅ Stored chunks 201 to 300
✅ Stored chunks 301 to 400
✅ Stored chunks 401 to 500
✅ Stored chunks 501 to 600
✅ Stored chunks 601 to 700
✅ Stored chunks 701 to 800
✅ Stored chunks 801 to 814

✅ All chunks stored in ChromaDB
Total vectors: 1001


#Create Retrieval Function

In [71]:
def retrieve_sports_context(query, sport=None, n_results=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).tolist()

    where_filter = None

    if sport:
        where_filter = {"sport": sport}

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where_filter,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    retrieved_items = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved_items.append({
            "text": document,
            "sport": metadata["sport"],
            "title": metadata["title"],
            "url": metadata["url"],
            "chunk_index": metadata["chunk_index"],
            "distance": distance
        })

    return retrieved_items

print("✅ Retrieval function ready")

✅ Retrieval function ready


#Test ChromaDB Retrieval


In [72]:
test_results = retrieve_sports_context(
    query="Which country won the first Cricket World Cup?",
    sport="Cricket",
    n_results=5
)

for index, result in enumerate(test_results, start=1):
    print(f"\n--- Result {index} ---")
    print("Title:", result["title"])
    print("Sport:", result["sport"])
    print("Distance:", round(result["distance"], 4))
    print("Source:", result["url"])
    print("Text:", result["text"][:500])


--- Result 1 ---
Title: Cricket World Cup
Sport: Cricket
Distance: 0.342
Source: https://en.wikipedia.org/wiki/Cricket_World_Cup
Text: he World Cup was reduced to fourteen. Australia lost their final group stage match against Pakistan on 19 March 2011, ending an unbeaten streak of 35 World Cup matches, which had begun on 23 May 1999. India won their second World Cup title by beating Sri Lanka by 6 wickets in the final at Wankhede Stadium in Mumbai, where the Indian captain M.S. Dhoni along with the spinning all-rounder Yuvraj Singh chased 275 with notable performances from Gautam Gambhir and Virat Kohli, making India the first 

--- Result 2 ---
Title: Cricket World Cup
Sport: Cricket
Distance: 0.3693
Source: https://en.wikipedia.org/wiki/Cricket_World_Cup
Text: out competition known as the Midlands Knock-Out Cup, and continuing with the inaugural Gillette Cup in 1963, one-day cricket grew in popularity in England. A national Sunday League was formed in 1969. The first One-Day Interna

#Test Multiple Sports Queries


In [73]:
test_queries = [
    {
        "sport": "Cricket",
        "query": "Which country won the first Cricket World Cup?"
    },
    {
        "sport": "Football",
        "query": "Which country won the first FIFA World Cup?"
    },
    {
        "sport": "Tennis",
        "query": "What are the four Grand Slam tournaments?"
    },
    {
        "sport": "Badminton",
        "query": "What is the Thomas Cup?"
    },
    {
        "sport": "Basketball",
        "query": "When was basketball invented?"
    }
]

for test in test_queries:
    print("\n" + "=" * 80)
    print("SPORT:", test["sport"])
    print("QUERY:", test["query"])

    results = retrieve_sports_context(
        query=test["query"],
        sport=test["sport"],
        n_results=3
    )

    for index, result in enumerate(results, start=1):
        print(f"\nResult {index}")
        print("Title:", result["title"])
        print("Distance:", round(result["distance"], 4))
        print("Text:", result["text"][:350])


SPORT: Cricket
QUERY: Which country won the first Cricket World Cup?

Result 1
Title: Cricket World Cup
Distance: 0.342
Text: he World Cup was reduced to fourteen. Australia lost their final group stage match against Pakistan on 19 March 2011, ending an unbeaten streak of 35 World Cup matches, which had begun on 23 May 1999. India won their second World Cup title by beating Sri Lanka by 6 wickets in the final at Wankhede Stadium in Mumbai, where the Indian captain M.S. Dh

Result 2
Title: Cricket World Cup
Distance: 0.3693
Text: out competition known as the Midlands Knock-Out Cup, and continuing with the inaugural Gillette Cup in 1963, one-day cricket grew in popularity in England. A national Sunday League was formed in 1969. The first One-Day International match was played on the fifth day of a rain-aborted Test match between England and Australia at Melbourne in 1971, to

Result 3
Title: Cricket World Cup
Distance: 0.3809
Text: n every tournament, five of which have won the title. T

#Create Readable Context Formatter

In [74]:
def format_retrieved_context(retrieved_items):
    formatted_sections = []

    for index, item in enumerate(retrieved_items, start=1):
        section = f"""
SOURCE {index}
Sport: {item['sport']}
Title: {item['title']}
URL: {item['url']}
Content:
{item['text']}
"""
        formatted_sections.append(section.strip())

    return "\n\n".join(formatted_sections)


print("✅ Context formatter ready")

✅ Context formatter ready


#Test Formatted Context


In [75]:
retrieved_items = retrieve_sports_context(
    query="History and winners of the Cricket World Cup",
    sport="Cricket",
    n_results=5
)

formatted_context = format_retrieved_context(retrieved_items)

print(formatted_context[:5000])

SOURCE 1
Sport: Cricket
Title: Cricket World Cup
URL: https://en.wikipedia.org/wiki/Cricket_World_Cup
Content:
out competition known as the Midlands Knock-Out Cup, and continuing with the inaugural Gillette Cup in 1963, one-day cricket grew in popularity in England. A national Sunday League was formed in 1969. The first One-Day International match was played on the fifth day of a rain-aborted Test match between England and Australia at Melbourne in 1971, to fill the time available and as compensation for the frustrated crowd. It was a forty over game with eight balls per over, and saw Australia win by 5 wickets. The success and popularity of the domestic one-day competitions in England and other parts of the world, as well as the early One-Day Internationals, prompted the ICC to consider organizing a Cricket World Cup. Prudential World Cups (1975–1983) The inaugural Cricket World Cup was hosted in 1975 by England, the only nation able to put forward the resources to stage an event of s

#Add Retrieval Quality Check




In [76]:
def filter_relevant_results(results, max_distance=0.90):
    relevant_results = [
        result
        for result in results
        if result["distance"] <= max_distance
    ]

    return relevant_results

In [77]:
results = retrieve_sports_context(
    query="Who won the first FIFA World Cup?",
    sport="Football",
    n_results=5
)

filtered_results = filter_relevant_results(
    results,
    max_distance=0.90
)

print("Retrieved:", len(results))
print("Relevant:", len(filtered_results))

for r in filtered_results:
    print(r["title"], round(r["distance"],4))

Retrieved: 5
Relevant: 5
FIFA World Cup 0.3951
FIFA World Cup 0.453
FIFA World Cup 0.464
FIFA World Cup 0.4771
FIFA World Cup 0.4788


#Install LangChain Text Splitter


In [78]:
!pip install -q langchain-text-splitters

#Import Recursive Text Splitter


In [79]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("✅ Text splitter imported")

✅ Text splitter imported


#Create Better Text Splitter


In [80]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

print("✅ Recursive text splitter ready")

✅ Recursive text splitter ready


#Recreate Sports Chunks


In [81]:
better_sports_chunks = []

for document in clean_documents:
    chunks = text_splitter.split_text(document["text"])

    for index, chunk in enumerate(chunks):
        better_sports_chunks.append({
            "id": f"{document['sport']}_{document['title']}_{index}",
            "sport": document["sport"],
            "title": document["title"],
            "url": document["url"],
            "chunk_index": index,
            "text": chunk.strip()
        })

print("✅ Better chunks created")
print("Total chunks:", len(better_sports_chunks))

✅ Better chunks created
Total chunks: 1001


#Replace Old Chunk Variable


In [82]:
sports_chunks = better_sports_chunks

print("✅ New chunks activated")
print("Active chunks:", len(sports_chunks))

✅ New chunks activated
Active chunks: 1001


#Reprepare Embedding Data


In [83]:
chunk_ids = [chunk["id"] for chunk in sports_chunks]
chunk_texts = [chunk["text"] for chunk in sports_chunks]

chunk_metadata = [
    {
        "sport": chunk["sport"],
        "title": chunk["title"],
        "url": chunk["url"],
        "chunk_index": chunk["chunk_index"]
    }
    for chunk in sports_chunks
]

print("✅ New embedding data prepared")
print("Total chunks:", len(chunk_texts))

✅ New embedding data prepared
Total chunks: 1001


#Generate New Embeddings


In [84]:
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("✅ New embeddings generated")
print("Shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

✅ New embeddings generated
Shape: (1001, 384)


#Delete Old ChromaDB Collection


In [85]:
try:
    chroma_client.delete_collection("sports_knowledge")
    print("✅ Old collection deleted")
except Exception as error:
    print("Collection deletion note:", error)

✅ Old collection deleted


#Create Fresh ChromaDB Collection


In [86]:
collection = chroma_client.create_collection(
    name="sports_knowledge",
    metadata={"hnsw:space": "cosine"}
)

print("✅ Fresh ChromaDB collection created")

✅ Fresh ChromaDB collection created


#Store New Embeddings


In [87]:
batch_size = 100

for start in range(0, len(chunk_ids), batch_size):
    end = min(start + batch_size, len(chunk_ids))

    collection.upsert(
        ids=chunk_ids[start:end],
        documents=chunk_texts[start:end],
        embeddings=chunk_embeddings[start:end].tolist(),
        metadatas=chunk_metadata[start:end]
    )

    print(f"✅ Stored {start + 1} to {end}")

print("\nTotal vectors:", collection.count())

✅ Stored 1 to 100
✅ Stored 101 to 200
✅ Stored 201 to 300
✅ Stored 301 to 400
✅ Stored 401 to 500
✅ Stored 501 to 600
✅ Stored 601 to 700
✅ Stored 701 to 800
✅ Stored 801 to 900
✅ Stored 901 to 1000
✅ Stored 1001 to 1001

Total vectors: 1001


#Retest Important Questions


In [88]:
accuracy_queries = [
    ("Cricket", "Who won the first Cricket World Cup?"),
    ("Football", "Who won the first FIFA World Cup?"),
    ("Tennis", "What are the four Grand Slam tournaments?"),
    ("Badminton", "What is the Thomas Cup?"),
    ("Basketball", "Who invented basketball?")
]

for sport, query in accuracy_queries:
    print("\n" + "=" * 80)
    print("SPORT:", sport)
    print("QUERY:", query)

    results = retrieve_sports_context(
        query=query,
        sport=sport,
        n_results=5
    )

    for index, result in enumerate(results, start=1):
        print(f"\nResult {index}")
        print("Title:", result["title"])
        print("Distance:", round(result["distance"], 4))
        print("Text:", result["text"][:600])


SPORT: Cricket
QUERY: Who won the first Cricket World Cup?

Result 1
Title: Cricket World Cup
Distance: 0.3808
Text: . A national Sunday League was formed in 1969. The first One-Day International match was played on the fifth day of a rain-aborted Test match between England and Australia at Melbourne in 1971, to fill the time available and as compensation for the frustrated crowd. It was a forty over game with eight balls per over, and saw Australia win by 5 wickets. The success and popularity of the domestic one-day competitions in England and other parts of the world, as well as the early One-Day Internationals, prompted the ICC to consider organizing a Cricket World Cup. Prudential World Cups (1975–1983) 

Result 2
Title: Cricket World Cup
Distance: 0.3857
Text: The ICC Men's Cricket World Cup is a quadrennial world cup for cricket in One Day International (ODI) format, organised by the International Cricket Council (ICC). The tournament is one of the world's most viewed sporting e

#Install Current SDKs


In [89]:
!pip install -q -U google-genai tavily-python

#Securely Enter API Keys

In [90]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass(
    "Enter your Gemini API key: "
).strip()

os.environ["TAVILY_API_KEY"] = getpass(
    "Enter your Tavily API key: "
).strip()

print("✅ API keys configured securely")

Enter your Gemini API key: ··········
Enter your Tavily API key: ··········
✅ API keys configured securely


#Configure Gemini Client


In [91]:
from google import genai

gemini_client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("✅ Gemini client configured")

✅ Gemini client configured


#Test Gemini Connection


In [92]:
response = gemini_client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Reply only with: Gemini connection successful"
)

print(response.text)

Gemini connection successful


#Configure Tavily Client


In [93]:
from tavily import TavilyClient

tavily_client = TavilyClient(
    api_key=os.environ["TAVILY_API_KEY"]
)

print("✅ Tavily client configured")

✅ Tavily client configured


#Test Tavily Search


In [94]:
search_result = tavily_client.search(
    query="latest cricket news",
    max_results=3
)

for index, result in enumerate(search_result["results"], start=1):
    print(f"\nResult {index}")
    print("Title:", result["title"])
    print("URL:", result["url"])
    print("Content:", result["content"][:300])


Result 1
Title: Cricket News Today | Live Cricket Score | Cricket Match Today Updates
URL: https://www.hindustantimes.com/cricket
Content: # Latest Cricket News. ### Sobers was one of a kind: His passing marks end to most glorious cricket epoch. Rohit Sharma has been struggling in the ongoing ODI series against England. England defeated India by 56 runs, completing their 4-0 rout and also taking the No. 1 spot in the T20I rankings. ENG

Result 2
Title: Cricket News - Latest and Breaking News on Cricket, Expert Reviews
URL: https://sports.ndtv.com/cricket/news
Content: * [1st ODI, India in England, 3 ODI Series, 2026 at Birmingham, Jul 14, 2026. Tue, Jul 14, 2026 - 3:30 PM IST](/cricket/eng-vs-ind-scorecard-live-cricket-score-india-in-england-3-odi-series-2026-1st-odi-enin07142026264917). Thu, Jul 16, 2026 - 5:30 PM IST](/cricket/eng-vs-ind-scorecard-live-cricket-

Result 3
Title: ICC Cricket News
URL: https://www.icc-cricket.com/news
Content: icc-w-t20-wc-2026. icc-women-s-t20i-challe

#Search Recent Sports Information


In [95]:
def search_recent_sports_information(query, max_results=3):
    """
    Search recent sports information using Tavily.
    """

    try:
        response = tavily_client.search(
            query=query,
            max_results=max_results
        )

        return response.get("results", [])

    except Exception as error:
        print(f"Tavily Error: {error}")
        return []

#Format Web Context


In [96]:
def format_web_context(results):
    """
    Convert Tavily search results into readable context.
    """

    if not results:
        return ""

    context = ""

    for index, item in enumerate(results, start=1):

        context += f"""
Source {index}

Title:
{item.get("title","")}

Content:
{item.get("content","")}

URL:
{item.get("url","")}

----------------------------------

"""

    return context

#Test


In [97]:
results = search_recent_sports_information(
    "latest cricket world cup news"
)

web_context = format_web_context(results)

print(web_context[:1000])


Source 1

Title:
Cricket World Cup - latest news, breaking stories and comment - The Independent

Content:
# Cricket World Cup. <p>Australia won the last 50-over World Cup in 2023</p>. <p>Australia are the defending ODI World Cup champions</p>. <p>Thomas Rew has led England into the final</p>. <p>Ben Mayes broke multiple records at the Under-19 Cricket World Cup on Wednesday</p>. <p>India won the Women's Cricket World Cup for the first time on Sunday</p>. <p>Marizanne Kapp of South Africa celebrates with teammates Laura Wolvaardt and Sinalo Jafta after taking the wicket of Charlie Dean of England during the ICC Women's Cricket World Cup India 2025 semi-final match on 29 October 2025 in Guwahati, India. <p>England suffered a 125-run semi-final defeat to South Africa</p>. ## Dominant South Africa beat England to reach Cricket World Cup final. <p>England’s Linsey Smith celebrates the dismissal of South Africa captain Laura Wolvaardt (left), during their ICC Women’s Cricket World Cup matc

#Quiz Generation Function

In [98]:
def generate_quiz(sport, difficulty):

    # Step 1: Retrieve Wikipedia context
    rag_context = format_retrieved_context(
        retrieve_sports_context(
            query=f"{sport} {difficulty}",
            sport=sport,
            n_results=5
        )
    )

    # Step 2: Retrieve latest web context
    web_results = search_recent_sports_information(
        f"Latest {sport} news"
    )

    web_context = format_web_context(web_results)

    # Step 3: Create prompt
    prompt = f"""
You are an expert sports quiz generator.

Generate exactly 5 multiple-choice questions.

Sport:
{sport}

Difficulty:
{difficulty}

Wikipedia Knowledge:
{rag_context}

Latest Sports News:
{web_context}

Rules:
1. Generate exactly 5 MCQs.
2. Each question must have exactly four options: A, B, C and D.
3. Provide one correct answer and a brief explanation.
4. Use only facts explicitly supported by the supplied Wikipedia or Tavily context.
5. Do not use outside knowledge, assumptions, or unsupported claims.
6. Use recent news only when the Tavily context clearly contains the fact.
7. Otherwise generate timeless questions from the Wikipedia context.
8. Match the requested difficulty level.
9. Avoid ambiguous questions and duplicate questions.
10. Return clean Markdown.
"""

    # Step 4: Generate quiz using Gemini
    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )

    # Step 5: Clean output
    quiz = response.text

    quiz = quiz.replace("&nbsp;", " ")
    quiz = quiz.replace("###", "##")

    return quiz

#Test Quiz


In [99]:
quiz = generate_quiz(
    sport="Cricket",
    difficulty="Medium"
)

print(quiz)

## Question 1

Under the rules of cricket, what is the maximum permitted width for the blade of a cricket bat?

A) 3.8 inches (9.7 cm)  
B) 4.25 inches (10.8 cm)  
C) 4.75 inches (12.1 cm)  
D) 5.25 inches (13.3 cm)  

**Correct Answer:** B  
**Explanation:** According to Source 1, the blade of a cricket bat must not be more than 4.25 inches (10.8 cm) wide.

---

## Question 2

While the International Cricket Council (ICC) is the global governing body of cricket, which organization maintains the official "Laws of Cricket"?

A) The International Cricket Council (ICC)  
B) The Marylebone Cricket Club (MCC)  
C) The Board of Control for Cricket in India (BCCI)  
D) The Association of Cricket Officials (ACO)  

**Correct Answer:** B  
**Explanation:** Source 2 states that the game's rules, the Laws of Cricket, are maintained by the Marylebone Cricket Club (MCC) in London.

---

## Question 3

The phrase "on a sticky wicket" is a common English metaphor for a difficult circumstance. What ph

#Create Dashboard Function


In [100]:
def generate_sports_quiz(sport, difficulty):
    return generate_quiz(sport, difficulty)

#Gradio Dashboard


In [101]:
import gradio as gr

sports = [
    "Cricket",
    "Football",
    "Tennis",
    "Badminton",
    "Basketball"
]

difficulty_levels = [
    "Easy",
    "Medium",
    "Hard"
]

demo = gr.Interface(
    fn=generate_sports_quiz,

    inputs=[
        gr.Dropdown(
            choices=sports,
            value="Cricket",
            label="Select Sport"
        ),

        gr.Dropdown(
            choices=difficulty_levels,
            value="Medium",
            label="Select Difficulty"
        )
    ],

    outputs=gr.Markdown(label="Generated Quiz"),

    title="🏆 AI Powered Sports Quiz Generator",

    description="""
Generate sports quizzes using

• Wikipedia Knowledge
• ChromaDB Retrieval
• Tavily Web Search
• Gemini 3.5 Flash
"""
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b805b800b760b5379e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
